In [1]:

import polars as pl
fcs= pl.read_parquet("forecast_result.parquet")
daa = pl.read_parquet("daa_price.parquet")
daa = daa.select(pl.col("delivery_start").dt.convert_time_zone("UTC"),pl.col("daa_price"))

result =(
    daa
    .join(fcs, on="delivery_start", how="full",coalesce=True)

) 
print("DAA Prices and Residual Load Forecast")
display(result)

DAA Prices and Residual Load Forecast


delivery_start,daa_price,entsoe,enappsys,energy_weather,GFS_ENS,EC_OP,EC_46,GFS_OP,EC_AIFS,EC_ENS,EC_AIFS_ENS,tech_avail
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-01-01 00:00:00 UTC,60.01,8469.0,10310.0,null,6870.44,6489.05,10183.42,3739.45,8379.26,7036.98,null,-2660.471509
2026-01-01 00:15:00 UTC,48.9,7208.0,9457.0,null,6180.66,6216.17,10183.42,3088.67,7693.09,6318.96,null,-3729.071544
2026-01-01 00:30:00 UTC,36.74,6298.0,9083.0,null,5369.82,6189.5,10183.42,2499.47,6412.5,5401.69,null,-4641.857449
2026-01-01 00:45:00 UTC,23.93,5804.0,8311.0,null,4377.39,5455.32,10183.42,2295.28,6513.04,4935.19,null,-5460.320099
2026-01-01 01:00:00 UTC,36.01,4976.0,8055.0,null,3175.97,5328.14,7598.38,2008.71,5514.48,4834.84,null,-6194.019856
…,…,…,…,…,…,…,…,…,…,…,…,…
2026-08-31 23:15:00 UTC,146.05,null,null,null,null,null,null,null,null,null,null,17403.638161
2026-08-31 23:30:00 UTC,146.19,null,null,null,5529.26,11703.08,10376.89,5542.55,14044.53,8583.28,11692.38,null
2026-08-31 23:30:00 UTC,146.19,null,null,null,null,null,null,null,null,null,null,17548.877713


In [2]:
# fit a model : Random Forest Feature Importance(including techavail)
#When used within an statistical model to predict DAA prices, which one has the highest feature importance?

from sklearn.ensemble import RandomForestRegressor
import numpy as np

features = result.columns[2:]

df = (
    result
    .select(features + ["daa_price"])
    .drop_nulls()
)

X = df.select(features).to_numpy()
y = df["daa_price"].to_numpy()

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X, y)

importance = pl.DataFrame({
    "forecast": features,
    "importance": rf.feature_importances_
}).sort("importance", descending=True)

print(importance)

shape: (11, 2)
┌────────────────┬────────────┐
│ forecast       ┆ importance │
│ ---            ┆ ---        │
│ str            ┆ f64        │
╞════════════════╪════════════╡
│ tech_avail     ┆ 0.628451   │
│ energy_weather ┆ 0.136361   │
│ enappsys       ┆ 0.070256   │
│ entsoe         ┆ 0.031554   │
│ EC_ENS         ┆ 0.02897    │
│ …              ┆ …          │
│ GFS_OP         ┆ 0.0191     │
│ GFS_ENS        ┆ 0.0184     │
│ EC_AIFS_ENS    ┆ 0.015402   │
│ EC_46          ┆ 0.015313   │
│ EC_AIFS        ┆ 0.010659   │
└────────────────┴────────────┘


In [3]:
# fit a model : Random Forest Feature Importance(excluding techavail)
#When used within an statistical model to predict DAA prices, which one has the highest feature importance?

from sklearn.ensemble import RandomForestRegressor
import numpy as np

features = result.columns[2:-1]

df = (
    result
    .select(features + ["daa_price"])
    .drop_nulls()
)

X = df.select(features).to_numpy()
y = df["daa_price"].to_numpy()

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X, y)

importance = pl.DataFrame({
    "forecast": features,
    "importance": rf.feature_importances_
}).sort("importance", descending=True)

print(importance)

shape: (10, 2)
┌────────────────┬────────────┐
│ forecast       ┆ importance │
│ ---            ┆ ---        │
│ str            ┆ f64        │
╞════════════════╪════════════╡
│ energy_weather ┆ 0.732455   │
│ enappsys       ┆ 0.085248   │
│ EC_ENS         ┆ 0.037711   │
│ entsoe         ┆ 0.032943   │
│ GFS_ENS        ┆ 0.024814   │
│ EC_OP          ┆ 0.022653   │
│ GFS_OP         ┆ 0.021645   │
│ EC_46          ┆ 0.016942   │
│ EC_AIFS_ENS    ┆ 0.01301    │
│ EC_AIFS        ┆ 0.01258    │
└────────────────┴────────────┘


In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import polars as pl

providers = [
    "entsoe",
    "enappsys",
    "energy_weather",
    "EC_ENS"
]

# --------------------------------------------------
# 1. RANDOM FOREST FEATURE IMPORTANCE
# Same observations for all 4 providers
# --------------------------------------------------

rf_clean = (
    result
    .select(["daa_price"] + providers)
    .drop_nulls()
)

X = rf_clean.select(providers).to_numpy()
y = rf_clean["daa_price"].to_numpy()

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

importance = pl.DataFrame({
    "forecast": providers,
    "importance": rf.feature_importances_
})


# --------------------------------------------------
# 2. INDIVIDUAL LINEAR REGRESSION FOR EACH PROVIDER
# --------------------------------------------------

reg_results = []

for provider in providers:

    clean = (
        result
        .select(["daa_price", provider])
        .drop_nulls()
    )

    X = clean.select(provider).to_numpy()
    y = clean["daa_price"].to_numpy()

    model = LinearRegression()
    model.fit(X, y)

    pred = model.predict(X)

    mae = mean_absolute_error(y, pred)
    mse = mean_squared_error(y, pred)
    rmse = np.sqrt(mse)
    r2 = model.score(X, y)

    reg_results.append({
        "forecast": provider,
        "N": clean.height,
        "coefficient": model.coef_[0],
        "R2": r2,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse
    })

reg_results = pl.DataFrame(reg_results)


# --------------------------------------------------
# 3. COMBINE EVERYTHING
# --------------------------------------------------

comparison = (
    importance
    .join(
        reg_results,
        on="forecast",
        how="left"
    )
    .sort("importance", descending=True)
)

print(f"Random Forest observations: {rf_clean.height:,}")
display(comparison)

Random Forest observations: 12,944


forecast,importance,N,coefficient,R2,MAE,MSE,RMSE
str,f64,i64,f64,f64,f64,f64,f64
"""energy_weather""",0.752009,19304,0.002845,0.674664,25.102162,1343.708375,36.656628
"""enappsys""",0.110083,24625,0.002428,0.518959,29.857119,1784.219449,42.240022
"""EC_ENS""",0.081415,25444,0.002684,0.597719,26.95846,1519.864541,38.98544
"""entsoe""",0.056494,17264,0.002542,0.492152,30.405334,1694.858652,41.168661
